# Merge opponent data

The matches file (`matches_premier-league_2022-23.xlsx`) is **match-level**: one row per fixture, split into home/away.
The teams file (`PL Teams 2022--23 Dataset.csv`) is **team-match level**: one row per team per matchday.

Plan: reshape the matches file from wide (home/away) to long (one row per team per match), so every team-match gets its opponent and the opponent's stats. Then merge onto the teams file by `team_name` + `matchday`.

In [1]:
import pandas as pd

In [2]:
# Load both datasets
teams = pd.read_csv("PL Teams 2022--23 Dataset.csv")

# The Excel file has many junk 'Unnamed' and lineup columns; keep only what we need
use_cols = [
    'stage', 'date', 'team_name_home', 'team_name_away',
    'team_home_score', 'team_away_score',
    'possession_home', 'possession_away',
    'total_shots_home', 'total_shots_away',
    'shots_on_target_home', 'shots_on_target_away',
    'duels_won_home', 'duels_won_away', 'location',
]
matches = pd.read_excel("matches_premier-league_2022-23.xlsx")[use_cols]

# Drop any empty rows
matches = matches.dropna(subset=['team_name_home', 'team_name_away']).reset_index(drop=True)
print('teams:', teams.shape, ' matches:', matches.shape)
matches.head()

teams: (760, 8)  matches: (380, 15)


,stage,date,team_name_home,team_name_away,team_home_score,team_away_score,possession_home,possession_away,total_shots_home,total_shots_away,shots_on_target_home,shots_on_target_away,duels_won_home,duels_won_away,location
0,Matchday 1,2022-08-06,AFC Bournemouth,Aston Villa,2.0,0.0,0.34,0.66,7.0,15,3.0,2.0,0.52,0.48,Vitality Stadium
1,Matchday 3,2022-08-20,AFC Bournemouth,Arsenal,0.0,3.0,0.41,0.59,6.0,14,1.0,6.0,0.50,0.51,Vitality Stadium
2,Matchday 5,2022-08-31,AFC Bournemouth,Wolverhampton Wanderers,0.0,0.0,0.34,0.66,5.0,17,2.0,4.0,0.51,0.49,Vitality Stadium
3,Matchday 9,2022-10-01,AFC Bournemouth,Brentford,0.0,0.0,0.42,0.58,7.0,13,1.0,4.0,0.48,0.52,Vitality Stadium
4,Matchday 10,2022-10-08,AFC Bournemouth,Leicester City,2.0,1.0,0.44,0.56,10.0,9,4.0,5.0,0.42,0.58,Vitality Stadium


In [3]:
# --- Clean up the matches file ---
# 1) parse matchday number out of 'stage' (e.g. 'Matchday 1' -> 1)
matches['matchday'] = matches['stage'].str.extract(r'(\d+)').astype(int)

# 2) normalise the two team names that differ between the files
name_fix = {
    'AFC Bournemouth': 'Bournemouth',
    'Leicester City': 'Leicester',
}
matches['team_name_home'] = matches['team_name_home'].replace(name_fix)
matches['team_name_away'] = matches['team_name_away'].replace(name_fix)

matches[['stage', 'matchday', 'team_name_home', 'team_name_away']].head()

,stage,matchday,team_name_home,team_name_away
0,Matchday 1,1,Bournemouth,Aston Villa
1,Matchday 3,3,Bournemouth,Arsenal
2,Matchday 5,5,Bournemouth,Wolverhampton Wanderers
3,Matchday 9,9,Bournemouth,Brentford
4,Matchday 10,10,Bournemouth,Leicester


In [4]:
# --- Reshape wide (home/away) -> long (one row per team per match) ---
# Home perspective: the team is the home side, the opponent is the away side.
home = matches.rename(columns={
    'team_name_home': 'team_name',
    'team_name_away': 'opponent',
    'team_away_score': 'opponent_goals',
    'possession_away': 'opponent_possession',
    'total_shots_away': 'opponent_total_shots',
    'shots_on_target_away': 'opponent_shots_on_target',
    'duels_won_away': 'opponent_duels_won',
})[['matchday', 'date', 'team_name', 'opponent', 'opponent_goals',
    'opponent_possession', 'opponent_total_shots',
    'opponent_shots_on_target', 'opponent_duels_won']]
home['is_home'] = 1

# Away perspective: the team is the away side, the opponent is the home side.
away = matches.rename(columns={
    'team_name_away': 'team_name',
    'team_name_home': 'opponent',
    'team_home_score': 'opponent_goals',
    'possession_home': 'opponent_possession',
    'total_shots_home': 'opponent_total_shots',
    'shots_on_target_home': 'opponent_shots_on_target',
    'duels_won_home': 'opponent_duels_won',
})[['matchday', 'date', 'team_name', 'opponent', 'opponent_goals',
    'opponent_possession', 'opponent_total_shots',
    'opponent_shots_on_target', 'opponent_duels_won']]
away['is_home'] = 0

# Stack them: 380 matches x 2 perspectives = 760 team-match rows
long_matches = pd.concat([home, away], ignore_index=True)
print('long_matches:', long_matches.shape)
long_matches.head()

long_matches: (760, 10)


,matchday,date,team_name,opponent,opponent_goals,opponent_possession,opponent_total_shots,opponent_shots_on_target,opponent_duels_won,is_home
0,1,2022-08-06,Bournemouth,Aston Villa,0.0,0.66,15.0,2.0,0.48,1
1,3,2022-08-20,Bournemouth,Arsenal,3.0,0.59,14.0,6.0,0.51,1
2,5,2022-08-31,Bournemouth,Wolverhampton Wanderers,0.0,0.66,17.0,4.0,0.49,1
3,9,2022-10-01,Bournemouth,Brentford,0.0,0.58,13.0,4.0,0.52,1
4,10,2022-10-08,Bournemouth,Leicester,1.0,0.56,9.0,5.0,0.58,1


In [5]:
# --- Merge opponent info onto the teams dataset ---
# Join key: team_name + matchday. Left join keeps every original team row.
merged = teams.merge(long_matches, on=['team_name', 'matchday'], how='left')

print('merged:', merged.shape)
# Any team-match rows that failed to find an opponent?
missing = merged['opponent'].isna().sum()
print('rows with no opponent matched:', missing)
merged.head(10)

merged: (760, 16)
rows with no opponent matched: 0


,table_position,team_name,matchday,total_shots,shots_on_target,goals,possesion,duels_won,date,opponent,opponent_goals,opponent_possession,opponent_total_shots,opponent_shots_on_target,opponent_duels_won,is_home
0,1,Manchester City,1,8,3,2,0.57,0.57,2022-08-07,West Ham United,0.0,0.24,6.0,1.0,0.43,0
1,1,Manchester City,2,19,5,4,0.67,0.52,2022-08-13,Bournemouth,0.0,0.33,3.0,1.0,0.48,1
2,1,Manchester City,3,16,4,3,0.69,0.55,2022-08-21,Newcastle United,3.0,0.31,12.0,6.0,0.45,0
3,1,Manchester City,4,18,5,4,0.75,0.56,2022-08-27,Crystal Palace,2.0,0.25,2.0,2.0,0.45,1
4,1,Manchester City,5,17,8,6,0.75,0.53,2022-08-31,Nottingham Forest,0.0,0.25,8.0,1.0,0.47,1
5,1,Manchester City,6,9,3,1,0.35,0.60,2022-09-03,Aston Villa,1.0,0.27,3.0,1.0,0.41,0
6,1,Manchester City,7,16,5,4,0.73,0.52,2023-01-19,Tottenham Hotspur,2.0,0.42,9.0,3.0,0.48,1
7,1,Manchester City,8,14,5,3,0.60,0.52,2022-09-17,Wolverhampton Wanderers,0.0,0.40,6.0,1.0,0.48,0
8,1,Manchester City,9,22,8,6,0.81,0.62,2022-10-02,Manchester United,3.0,0.46,12.0,8.0,0.38,1
9,1,Manchester City,10,17,6,4,0.66,0.54,2022-10-08,Southampton,0.0,0.34,5.0,0.0,0.46,1


In [6]:
# Sanity check: a team's goals here should match the opponent's 'opponent_goals'
# recorded from the other side of the same fixture.
print(merged[['team_name', 'matchday', 'opponent', 'is_home', 'goals',
              'opponent_goals', 'opponent_shots_on_target']].head(12).to_string())

          team_name  matchday                 opponent  is_home  goals  opponent_goals  opponent_shots_on_target
0   Manchester City         1          West Ham United        0      2             0.0                       1.0
1   Manchester City         2              Bournemouth        1      4             0.0                       1.0
2   Manchester City         3         Newcastle United        0      3             3.0                       6.0
3   Manchester City         4           Crystal Palace        1      4             2.0                       2.0
4   Manchester City         5        Nottingham Forest        1      6             0.0                       1.0
5   Manchester City         6              Aston Villa        0      1             1.0                       1.0
6   Manchester City         7        Tottenham Hotspur        1      4             2.0                       3.0
7   Manchester City         8  Wolverhampton Wanderers        0      3             0.0          

In [7]:
# Save the merged dataset for the data-preparation / modelling notebooks
merged.to_csv("merged_with_opponent.csv", index=False)
print('saved merged_with_opponent.csv', merged.shape)

saved merged_with_opponent.csv (760, 16)
